# Accept Parameters


In [5]:
# @title tags {"display-mode":"code"}
#@title { tags: ["parameters"] }
dataset_path  = "blah.csv";
if not dataset_path :
    raise ValueError("dataset_path must be provided!")
print("dataset path:", dataset_path)

dataset path: blah.csv


# Install the packages


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')



# !pip install --cache-dir /content/drive/MyDrive/pip_cache  -q boto3
# !pip install --cache-dir /content/drive/MyDrive/pip_cache -q pandas numpy scikit-learn jupyter tensorflow keras
# !pip install --cache-dir /content/drive/MyDrive/pip_cache -q myanmartools icu-tokenizer

## Import Libraries

In [ ]:
# Import libraries and setup environment
import pandas as pd
import numpy as np
import json
import joblib
import os
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


# TensorFlow/Keras imports
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Embedding, Dropout, Bidirectional
from keras.utils import pad_sequences, to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
try:
    from keras.layers import TextVectorization
    print("✅ Using keras.layers.TextVectorization")
except ImportError:
  print("Error when importing keras.layers.TextVectorization")
  from tf.keras.layers import TextVectorization
  print("✅ Using tf.keras.layers.TextVectorization")
except Exception as e:
    print(f"Error when importing keras.layers.TextVectorization: {e}")


# IPython display imports with fallback
try:
    from IPython.display import display, Markdown
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False
    def display(x):
        print(x)
    def Markdown(text):
        return text.replace('#', '').replace('*', '')


import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print("🚀 Deep Learning Environment Setup Complete!")
if IPYTHON_AVAILABLE:
    print("✅ IPython display available")
else:
    print("⚠️ IPython display not available - using print fallback")



✅ Using keras.layers.TextVectorization
🚀 Deep Learning Environment Setup Complete!
✅ IPython display available


# Helper Functions for data preprocessing



In [ ]:
def convert_zawgyi_to_unicode(text):
    #detect the text is Zawgyi or Unicode
    if isinstance(text,str) and detector.get_zawgyi_probability(text) > 0.9:
        return converter.transliterate(text)
    return text


#StopWords

stopWords = [
    "ပါ", "က", "တယ်", "ကို", "နေ", "တာ", "သည်", "မှာ", "တွေ", "ရ",
    "တော့", "ဖြစ်", "တဲ့", "များ", "ပြီး", "လည်း", "နဲ့",
    "ဆို", "ပဲ", "သူ", "ခဲ့", "ဘူး", "တစ်", "လိုက်", "မှ", "ကြ", "လာ", "သွား",
    "ပြော", "ပေး", "၏", "လား", "ရင်", "နိုင်", "ထား", "သော", "လေ", "မယ်",
    "လေး", "ပြီ", "ချင်", "တွင်", "လဲ", "နှင့်", "ဟုတ်",
     "ရဲ့", "နော်", "ဘာ", "ရေး", "ဒီ", "မှု", "လဲ", "ဟာ",
    "အရမ်း", "ပြန်", "နှစ်", "ပါစေ", "ဖို့", "ပြီ", "လား", "ခု", "ဖို့", "ရေ", "ဦး", "အောင်", "ရောက်", "ဘာ", "ဟု",
    "နောက်", "ကဲ", "မူ", "အတွက်", "ကြီး", "ထဲ", "ခြင်း", "စေ", "သေး", "မည်","မိ",
    "၍", "ဟယ်", "ရှင့်", "ဟင်", "။", "၊"
]

# Apply the mapping to 'final_label'
sentiment_map = {
    "Neutral": 0,
    "Postive": 1,
    "Positive": 1,  # In case of typo
    "Negative": -1
}

def remove_emoji_and_special_char(text):
    # Remove invisible characters
    text = re.sub(r'[\u200B\u200C\u200D\u2060\uFEFF]', '', text)

    # Remove corrupted or placeholder characters
    text = re.sub(r'[\uFFFD\u25A1\u25A0]', '', text)

    # Remove all characters except Burmese letters, diacritics, English letters, space, and essential punctuation
    text = re.sub(
        r'[^a-zA-Z\u1000-\u109F\u102C\u102D\u102E\u1036\u1039\u103A\u103C\u103D\u1030\u1031\u1032\u1037\u1038\s\'\"\[\],]', '',
        text
    )
     # Remove English words
    text = re.sub(r'[a-zA-Z]+', '', text)

    # Remove digits and other symbols
    text = re.sub(r'[0-9]', '', text)       # Remove numbers
    # Remove extra symbols
    text = re.sub(r'[\^\@\#\$\%\&\*\~\=\+\|\{\}\<\>\`\!]', '', text)

    # Add separator between Burmese and English words
    text = re.sub(r'([a-zA-Z]+)([က-အ])', r'\1,\2', text)
    text = re.sub(r'([က-အ])([a-zA-Z]+)', r'\1,\2', text)

    # Remove punctuation
    text = re.sub(r'[:;!?\\\(\)\.]', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove emojis
    emoji_pattern = re.compile(
        "[" +
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub('', text)

    return text

# Data Preprocessing
## Loading dataset from S3



In [ ]:
import pandas as pd
import boto3
from google.colab import userdata
userdata.get('AWS_REGION')

s3 = boto3.client('s3',
                   aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
                   aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'),
                   region_name=userdata.get('AWS_REGION'))
obj = s3.get_object(Bucket='myansen-dataset', Key='sample_data - Sheet1.csv') #"user.email+date.csv"`
df = pd.read_csv(obj['Body'])


print(df)

ModuleNotFoundError: No module named 'boto3'

## Converting Zawgyi to Unicode and Tokenize Text


In [ ]:

from myanmartools import ZawgyiDetector
from icu import Transliterator # !Tokenzier tools
from icu_tokenizer import Tokenizer

tokenizer = Tokenizer(lang='my')
detector = ZawgyiDetector()
converter = Transliterator.createInstance('Zawgyi-my')


#Convert the text in the 'original_comments' column
df["original_text"] = df["text"].apply(convert_zawgyi_to_unicode)
df["cleaned_text"] = df["original_text"].apply(
    lambda x: tokenizer.tokenize(x)
)
df.head()

,text,label,original_text,cleaned_text
0,ဒီဟာ အရမ်းကောင်းတယ် ကျေနပ်ပါတယ်,positive,ဒီဟာ အရမ်းကောင်းတယ် ကျေနပ်ပါတယ်,"[ဒီ, ဟာ, အရမ်း, ကောင်း, တယ်, ကျေနပ်, ပါ, တယ်]"
1,ဝန်ဆောင်မှု မကောင်းဘူး စိတ်ပျက်စရာ,negative,ဝန်ဆောင်မှု မကောင်းဘူး စိတ်ပျက်စရာ,"[ဝန်ဆောင်မှု, မ, ကောင်း, ဘူး, စိတ်ပျက်, စရာ]"
2,ပုံမှန်ပဲ ထူးခြားမှု မရှိဘူး,neutral,ပုံမှန်ပဲ ထူးခြားမှု မရှိဘူး,"[ပုံ, မှန်, ပဲ, ထူးခြား, မှု, မ, ရှိ, ဘူး]"
3,အစားအသောက်တွေ အရသာရှိတယ်,positive,အစားအသောက်တွေ အရသာရှိတယ်,"[အစားအသောက်, တွေ, အရသာ, ရှိ, တယ်]"
4,စျေးနှုန်း သိပ်မြင့်တယ် မတန်ဘူး,negative,စျေးနှုန်း သိပ်မြင့်တယ် မတန်ဘူး,"[စျေး, နှုန်း, သိပ်, မြင့်, တယ်, မတန်, ဘူး]"


## StopWord Removal

In [ ]:
# StopWord Removal
import re
df["cleaned_text"] = df["cleaned_text"].apply(
    lambda x: [
        word for cleaned_word in [remove_emoji_and_special_char(word) for word in x if isinstance(word, str)]
        for word in cleaned_word.split(',') # Split concatenated words (e.g., EnglishBurmese)
        if word not in stopWords and word.strip() != ''
    ]
)


df[["cleaned_text"]].head(20)

,cleaned_text
0,"[ကောင်း, ကျေနပ်]"
1,"[ဝန်ဆောင်မှု, မ, ကောင်း, စိတ်ပျက်, စရာ]"
2,"[ပုံ, မှန်, ထူးခြား, မ, ရှိ]"
3,"[အစားအသောက်, အရသာ, ရှိ]"
4,"[စျေး, နှုန်း, သိပ်, မြင့်, မတန်]"
5,"[အိမ်, ပြန်လာ]"


## Mapping Sentiment label with numberical format

In [ ]:
pd.set_option('future.no_silent_downcasting', True)
df["final_label"] = df["label"].replace(sentiment_map)
df

,text,label,original_text,cleaned_text,final_label
0,ဒီဟာ အရမ်းကောင်းတယ် ကျေနပ်ပါတယ်,positive,ဒီဟာ အရမ်းကောင်းတယ် ကျေနပ်ပါတယ်,"[ကောင်း, ကျေနပ်]",1
1,ဝန်ဆောင်မှု မကောင်းဘူး စိတ်ပျက်စရာ,negative,ဝန်ဆောင်မှု မကောင်းဘူး စိတ်ပျက်စရာ,"[ဝန်ဆောင်မှု, မ, ကောင်း, စိတ်ပျက်, စရာ]",-1
2,ပုံမှန်ပဲ ထူးခြားမှု မရှိဘူး,neutral,ပုံမှန်ပဲ ထူးခြားမှု မရှိဘူး,"[ပုံ, မှန်, ထူးခြား, မ, ရှိ]",0
3,အစားအသောက်တွေ အရသာရှိတယ်,positive,အစားအသောက်တွေ အရသာရှိတယ်,"[အစားအသောက်, အရသာ, ရှိ]",1
4,စျေးနှုန်း သိပ်မြင့်တယ် မတန်ဘူး,negative,စျေးနှုန်း သိပ်မြင့်တယ် မတန်ဘူး,"[စျေး, နှုန်း, သိပ်, မြင့်, မတန်]",-1
5,No 39 ပြီးရင်အိမ်ပြန်လာခဲ့တော့မယ်,neutral,No 39 ပြီးရင်အိမ်ပြန်လာခဲ့တော့မယ်,"[အိမ်, ပြန်လာ]",0


# Training With Baseline ML methods
## Helper Functions for training baseline model



# Splitting data into train and test

In [ ]:
#Change (,)separated into space separated becoz of keras only accepts space separeted string
df["cleaned_text_str"] = df["cleaned_text"].apply(lambda tokens: ' '.join(tokens))


# Select features and target
X = df['cleaned_text_str'].astype(str)
y = df['final_label']

# Dataset statistics
print(f"\n📈 Label Distribution:")
label_counts = y.value_counts().sort_index()
for label, count in label_counts.items():
    percentage = (count / len(y)) * 100
    print(f"   {label:2}: {count:4} samples ({percentage:5.1f}%)")

# Encode labels for neural network (0, 1, 2 instead of -1, 0, 1)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(np.unique(y_encoded))

print(f"\n🔢 Label Encoding:")
print(f"   Number of classes: {num_classes}")
print(f"   Original -> Encoded mapping:")
for orig, enc in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"     {orig:2} -> {enc}")

if all(label_counts >= 2) and len(y_encoded) * 0.2 >= num_classes:
    print("✅ Using Stratified Split...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
else:
    print("⚠️ Not enough samples for stratified split. Using regular split...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42
    )

print(f"\n📊 Train/Test Split:")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")
print(f"   Split ratio: {len(X_train)/len(X):0.1%} train / {len(X_test)/len(X):0.1%} test")



print(f"\n✅ Data loading completed successfully!")


📈 Label Distribution:
   -1:    2 samples ( 33.3%)
    0:    2 samples ( 33.3%)
    1:    2 samples ( 33.3%)

🔢 Label Encoding:
   Number of classes: 3
   Original -> Encoded mapping:
     -1 -> 0
      0 -> 1
      1 -> 2
⚠️ Not enough samples for stratified split. Using regular split...

📊 Train/Test Split:
   Training samples: 4
   Test samples: 2
   Split ratio: 66.7% train / 33.3% test

✅ Data loading completed successfully!


## Text preprocessing for LSTM



In [ ]:
# Tokenization parameters
MAX_VOCAB_SIZE = 5000
MAX_SEQUENCE_LENGTH = 100
EMBEDDING_DIM = 100

print(f"⚙️ Tokenization Configuration:")
print(f"   Max vocabulary size: {MAX_VOCAB_SIZE}")
print(f"   Max sequence length: {MAX_SEQUENCE_LENGTH}")
print(f"   Embedding dimension: {EMBEDDING_DIM}")



text_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH,
    standardize=None,  # Important: you've already tokenized
    split='whitespace' # Since your tokens are joined by spaces
)

# Fit the vocabulary only on training data
text_vectorizer.adapt(X_train)

# Convert labels to one-hot format
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)


X_train_pad = text_vectorizer(X_train)
X_test_pad = text_vectorizer(X_test)


# Get vocab list
vocab = text_vectorizer.get_vocabulary()
word_index = dict(zip(vocab, range(len(vocab))))

# Actual vocab size
ACTUAL_VOCAB_SIZE = len(vocab)

print(f"Word Index: {word_index}")
print("✅ TextVectorization fit completed successfully!")

⚙️ Tokenization Configuration:
   Max vocabulary size: 5000
   Max sequence length: 100
   Embedding dimension: 100
Word Index: {'': 0, '[UNK]': 1, np.str_('ရှိ'): 2, np.str_('အိမ်'): 3, np.str_('အရသာ'): 4, np.str_('အစားအသောက်'): 5, np.str_('သိပ်'): 6, np.str_('မှန်'): 7, np.str_('မြင့်'): 8, np.str_('မတန်'): 9, np.str_('မ'): 10, np.str_('ပြန်လာ'): 11, np.str_('ပုံ'): 12, np.str_('နှုန်း'): 13, np.str_('ထူးခြား'): 14, np.str_('စျေး'): 15}
✅ TextVectorization fit completed successfully!


### Defining LSTM architecture

In [ ]:
def create_simple_lstm(vocab_size, embedding_dim, max_length, num_classes):
    """
    Simple LSTM model for text classification

    Architecture: Embedding → LSTM → Dense → Output
    """
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length, name='embedding'),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2, name='lstm'),
        Dense(32, activation='relu', name='dense_hidden'),
        Dropout(0.3, name='dropout'),
        Dense(num_classes, activation='softmax', name='output')
    ])
    return model

def create_bidirectional_lstm(vocab_size, embedding_dim, max_length, num_classes):
    """
    Bidirectional LSTM model for improved context understanding

    Architecture: Embedding → BiLSTM → Dense → Output
    """
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length,name='embedding'),
        Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2), name='bidirectional_lstm'),
        Dense(32, activation='relu', name='dense_hidden'),
        Dropout(0.3, name='dropout'),
        Dense(num_classes, activation='softmax', name='output')
    ])
    return model

def create_stacked_lstm(vocab_size, embedding_dim, max_length, num_classes):
    """
    Stacked LSTM model for deep sequential processing

    Architecture: Embedding → LSTM → LSTM → Dense → Output
    """
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length,name='embedding'),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=True, name='lstm_1'),
        LSTM(32, dropout=0.2, recurrent_dropout=0.2, name='lstm_2'),
        Dense(32, activation='relu', name='dense_hidden'),
        Dropout(0.3, name='dropout'),
        Dense(num_classes, activation='softmax', name='output')
    ])
    return model

def create_deep_lstm(vocab_size, embedding_dim, max_length, num_classes):
    """
    Deep LSTM with additional layers for complex pattern recognition

    Architecture: Embedding → LSTM → LSTM → LSTM → Dense → Dense → Output
    """
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length, name='embedding'),
        LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True, name='lstm_1'),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=True, name='lstm_2'),
        LSTM(32, dropout=0.2, recurrent_dropout=0.2, name='lstm_3'),
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.4, name='dropout_1'),
        Dense(32, activation='relu', name='dense_2'),
        Dropout(0.3, name='dropout_2'),
        Dense(num_classes, activation='softmax', name='output')
    ])

    return model


# Model configurations
models_config = {
    'Simple LSTM': create_simple_lstm,
    'Bidirectional LSTM': create_bidirectional_lstm,
    'Stacked LSTM': create_stacked_lstm,
    'Deep LSTM': create_deep_lstm
}

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 0.001

print(f"\n⚙️ Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {EPOCHS}")
print(f"  Early stopping patience: {PATIENCE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"\n✅ Model architectures defined successfully!")




⚙️ Training Configuration:
  Batch size: 32
  Max epochs: 50
  Early stopping patience: 10
  Learning rate: 0.001

✅ Model architectures defined successfully!


# Train and evaluate LSTM models

In [ ]:


# Store results
results = []
trained_models = {}
training_histories = {}

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a model"""
    print(f"\n🔄 Training {model_name}...")

    # Compile model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stopping, reduce_lr],
        verbose=0
    )

    # Make predictions
    y_train_pred = model.predict(X_train, verbose=0)
    y_test_pred = model.predict(X_test, verbose=0)

    # Convert predictions to class labels
    y_train_pred_classes = np.argmax(y_train_pred, axis=1)
    y_test_pred_classes = np.argmax(y_test_pred, axis=1)

    # Convert true labels back to class labels
    y_train_true = np.argmax(y_train, axis=1)
    y_test_true = np.argmax(y_test, axis=1)

    # Calculate metrics
    train_acc = accuracy_score(y_train_true, y_train_pred_classes)
    test_acc = accuracy_score(y_test_true, y_test_pred_classes)
    train_f1 = f1_score(y_train_true, y_train_pred_classes, average='macro')
    test_f1 = f1_score(y_test_true, y_test_pred_classes, average='macro')

    # Store results
    result = {
        'Model': model_name,
        'Train_Accuracy': train_acc,
        'Test_Accuracy': test_acc,
        'Train_F1_Macro': train_f1,
        'Test_F1_Macro': test_f1,
        'Overfitting_Gap_Acc': train_acc - test_acc,
        'Overfitting_Gap_F1': train_f1 - test_f1,
        'Epochs_Trained': len(history.history['loss']),
        'Best_Val_Loss': min(history.history['val_loss'])
    }

    results.append(result)
    trained_models[model_name] = model
    training_histories[model_name] = history

    print(f"  ✅ {model_name} completed:")
    print(f"     Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    print(f"     Train F1: {train_f1:.4f} | Test F1: {test_f1:.4f}")
    print(f"     Epochs: {len(history.history['loss'])} | Best Val Loss: {min(history.history['val_loss']):.4f}")

    return model, history



# Train all models
print("🚀 Starting LSTM model training...")
print("=" * 60)

for model_name, model_func in models_config.items():
    # Create model with correct vocabulary size
    print(f"Creating {model_name} with vocab_size={ACTUAL_VOCAB_SIZE}")
    model = model_func(ACTUAL_VOCAB_SIZE, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH, num_classes)

    # Train and evaluate
    trained_model, history = evaluate_model(
        model, X_train_pad, X_test_pad, y_train_cat, y_test_cat, model_name
    )


print("\n✅ All LSTM models trained successfully!")

🚀 Starting LSTM model training...
Creating Simple LSTM with vocab_size=16

🔄 Training Simple LSTM...
  ✅ Simple LSTM completed:
     Train Acc: 0.5000 | Test Acc: 0.0000
     Train F1: 0.2222 | Test F1: 0.0000
     Epochs: 11 | Best Val Loss: 1.1005
Creating Bidirectional LSTM with vocab_size=16

🔄 Training Bidirectional LSTM...
  ✅ Bidirectional LSTM completed:
     Train Acc: 0.5000 | Test Acc: 0.5000
     Train F1: 0.2222 | Test F1: 0.3333
     Epochs: 11 | Best Val Loss: 1.0977
Creating Stacked LSTM with vocab_size=16

🔄 Training Stacked LSTM...


  ✅ Stacked LSTM completed:
     Train Acc: 0.2500 | Test Acc: 0.5000
     Train F1: 0.1333 | Test F1: 0.3333
     Epochs: 11 | Best Val Loss: 1.0837
Creating Deep LSTM with vocab_size=16

🔄 Training Deep LSTM...
  ✅ Deep LSTM completed:
     Train Acc: 0.2500 | Test Acc: 0.5000
     Train F1: 0.1333 | Test F1: 0.3333
     Epochs: 11 | Best Val Loss: 1.1001

✅ All LSTM models trained successfully!


#Setup Table to show Results

In [ ]:
# Display results in beautiful tables
display(Markdown("# 📊 LSTM Model Results"))

results_df = pd.DataFrame(results)

# Training results table
display(Markdown("## 🏋️ Training Set Results"))
train_results = results_df[['Model', 'Train_Accuracy', 'Train_F1_Macro']].copy()
train_results.columns = ['Model', 'Accuracy', 'F1-Macro']
train_results = train_results.round(4)

# Style train results
def highlight_best_train(s):
    if s.name in ['Accuracy', 'F1-Macro']:
        is_max = s == s.max()
        return ['background-color: green' if v else '' for v in is_max]
    return ['' for _ in s]

styled_train = train_results.style.apply(highlight_best_train, axis=0)
display(styled_train)

# Test results table
display(Markdown("## 🎯 Test Set Results"))
test_results = results_df[['Model', 'Test_Accuracy', 'Test_F1_Macro']].copy()
test_results.columns = ['Model', 'Accuracy', 'F1-Macro']
test_results = test_results.round(4)

# Style test results
def highlight_best_test(s):
    if s.name in ['Accuracy', 'F1-Macro']:
        is_max = s == s.max()
        return ['background-color: blue' if v else '' for v in is_max]
    return ['' for _ in s]

styled_test = test_results.style.apply(highlight_best_test, axis=0)
display(styled_test)

# Best performers
best_test_acc = test_results.loc[test_results['Accuracy'].idxmax()]
best_test_f1 = test_results.loc[test_results['F1-Macro'].idxmax()]

print(f"🥇 Best Test Accuracy: {best_test_acc['Model']} ({best_test_acc['Accuracy']:.4f})")
print(f"🥇 Best Test F1-Macro: {best_test_f1['Model']} ({best_test_f1['F1-Macro']:.4f})")

# Overfitting analysis
display(Markdown("## ⚠️ Overfitting Analysis"))
overfitting_df = results_df[['Model', 'Overfitting_Gap_Acc', 'Overfitting_Gap_F1', 'Epochs_Trained']].round(4)
overfitting_df.columns = ['Model', 'Accuracy_Gap', 'F1_Macro_Gap', 'Epochs_Trained']

# Style overfitting table
def highlight_overfitting(s):
    if s.name in ['Accuracy_Gap', 'F1_Macro_Gap']:
        colors = []
        for v in s:
            if v > 0.1:
                colors.append('background-color: lightcoral')
            elif v > 0.05:
                colors.append('background-color: lightyellow')
            else:
                colors.append('background-color: green')
        return colors
    return ['' for _ in s]

styled_overfitting = overfitting_df.style.apply(highlight_overfitting, axis=0)
display(styled_overfitting)

print("\nColor Legend:")
print("🟢 Green: Low overfitting (gap < 0.05)")
print("🟡 Yellow: Moderate overfitting (gap 0.05-0.1)")
print("🔴 Red: High overfitting (gap > 0.1)")

# 📊 LSTM Model Results

## 🏋️ Training Set Results

,Model,Accuracy,F1-Macro
0,Simple LSTM,0.500000,0.222200
1,Bidirectional LSTM,0.500000,0.222200
2,Stacked LSTM,0.250000,0.133300
3,Deep LSTM,0.250000,0.133300


## 🎯 Test Set Results

,Model,Accuracy,F1-Macro
0,Simple LSTM,0.000000,0.000000
1,Bidirectional LSTM,0.500000,0.333300
2,Stacked LSTM,0.500000,0.333300
3,Deep LSTM,0.500000,0.333300


🥇 Best Test Accuracy: Bidirectional LSTM (0.5000)
🥇 Best Test F1-Macro: Bidirectional LSTM (0.3333)


## ⚠️ Overfitting Analysis

,Model,Accuracy_Gap,F1_Macro_Gap,Epochs_Trained
0,Simple LSTM,0.500000,0.222200,11
1,Bidirectional LSTM,0.000000,-0.111100,11
2,Stacked LSTM,-0.250000,-0.200000,11
3,Deep LSTM,-0.250000,-0.200000,11



Color Legend:
🟢 Green: Low overfitting (gap < 0.05)
🟡 Yellow: Moderate overfitting (gap 0.05-0.1)
🔴 Red: High overfitting (gap > 0.1)


In [ ]:
# Training History Analysis (Text-based due to matplotlib issue)
display(Markdown("# 📈 Training History Analysis"))

print("📊 Training History Summary:")
print("=" * 60)

for model_name, history in training_histories.items():
    print(f"\n🔹 {model_name}:")
    print(f"   Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"   Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
    print(f"   Final Training Loss: {history.history['loss'][-1]:.4f}")
    print(f"   Final Validation Loss: {history.history['val_loss'][-1]:.4f}")
    print(f"   Best Validation Loss: {min(history.history['val_loss']):.4f}")
    print(f"   Total Epochs: {len(history.history['loss'])}")

    # Show training progression (first, middle, last epochs)
    epochs = len(history.history['loss'])
    if epochs >= 3:
        mid_epoch = epochs // 2
        print(f"   Training Progress:")
        print(f"     Epoch 1: Train Acc={history.history['accuracy'][0]:.4f}, Val Acc={history.history['val_accuracy'][0]:.4f}")
        print(f"     Epoch {mid_epoch+1}: Train Acc={history.history['accuracy'][mid_epoch]:.4f}, Val Acc={history.history['val_accuracy'][mid_epoch]:.4f}")
        print(f"     Epoch {epochs}: Train Acc={history.history['accuracy'][-1]:.4f}, Val Acc={history.history['val_accuracy'][-1]:.4f}")

# Text-based comparison
print(f"\n📈 Model Performance Comparison:")
print("=" * 60)
test_accs = [results_df[results_df['Model'] == model]['Test_Accuracy'].iloc[0] for model in models_config.keys()]
test_f1s = [results_df[results_df['Model'] == model]['Test_F1_Macro'].iloc[0] for model in models_config.keys()]

for i, model_name in enumerate(models_config.keys()):
    print(f"{model_name:20} | Test Acc: {test_accs[i]:.4f} | Test F1: {test_f1s[i]:.4f}")

print(f"\n🏆 Best Performance:")
best_acc_idx = test_accs.index(max(test_accs))
best_f1_idx = test_f1s.index(max(test_f1s))
print(f"   Best Accuracy: {list(models_config.keys())[best_acc_idx]} ({max(test_accs):.4f})")
print(f"   Best F1-Macro: {list(models_config.keys())[best_f1_idx]} ({max(test_f1s):.4f})")

# Summary
display(Markdown("## 📋 Summary & Recommendations"))
print("Key Findings:")
print("=" * 50)

best_model_name = results_df.loc[results_df['Test_F1_Macro'].idxmax(), 'Model']
best_model_results = results_df[results_df['Model'] == best_model_name].iloc[0]

print(f"✅ Best LSTM Model: {best_model_name}")
print(f"   Test Accuracy: {best_model_results['Test_Accuracy']:.4f}")
print(f"   Test F1-Macro: {best_model_results['Test_F1_Macro']:.4f}")
print(f"   Overfitting Gap (F1): {best_model_results['Overfitting_Gap_F1']:.4f}")
print(f"   Training Epochs: {best_model_results['Epochs_Trained']}")

print("\nRecommendations:")
print("=" * 50)
print("1. ✅ LSTM models can capture sequential patterns in text")
print("2. ✅ Early stopping prevents overfitting effectively")
print("3. ✅ Bidirectional LSTMs often perform better than unidirectional")
print("4. ✅ Consider pre-trained embeddings for better performance")
print("5. ✅ Monitor validation loss to detect overfitting")

# 📈 Training History Analysis

📊 Training History Summary:

🔹 Simple LSTM:
   Final Training Accuracy: 0.5000
   Final Validation Accuracy: 0.0000
   Final Training Loss: 1.0809
   Final Validation Loss: 1.1289
   Best Validation Loss: 1.1005
   Total Epochs: 11
   Training Progress:
     Epoch 1: Train Acc=0.0000, Val Acc=0.0000
     Epoch 6: Train Acc=0.5000, Val Acc=0.0000
     Epoch 11: Train Acc=0.5000, Val Acc=0.0000

🔹 Bidirectional LSTM:
   Final Training Accuracy: 0.5000
   Final Validation Accuracy: 0.0000
   Final Training Loss: 1.0647
   Final Validation Loss: 1.1763
   Best Validation Loss: 1.0977
   Total Epochs: 11
   Training Progress:
     Epoch 1: Train Acc=0.2500, Val Acc=0.5000
     Epoch 6: Train Acc=0.5000, Val Acc=0.0000
     Epoch 11: Train Acc=0.5000, Val Acc=0.0000

🔹 Stacked LSTM:
   Final Training Accuracy: 0.2500
   Final Validation Accuracy: 0.0000
   Final Training Loss: 1.0943
   Final Validation Loss: 1.1340
   Best Validation Loss: 1.0837
   Total Epochs: 11
   Training Progress:
  

## 📋 Summary & Recommendations

Key Findings:
✅ Best LSTM Model: Bidirectional LSTM
   Test Accuracy: 0.5000
   Test F1-Macro: 0.3333
   Overfitting Gap (F1): -0.1111
   Training Epochs: 11

Recommendations:
1. ✅ LSTM models can capture sequential patterns in text
2. ✅ Early stopping prevents overfitting effectively
3. ✅ Bidirectional LSTMs often perform better than unidirectional
4. ✅ Consider pre-trained embeddings for better performance
5. ✅ Monitor validation loss to detect overfitting


### Save the current best model

In [ ]:
# Save model
best_model = trained_models[best_model_name]
model_path = f"{best_model_name.replace(' ', '_').lower()}_best_model.h5"
best_model.save(model_path)

print(f"✅ Best model saved at: {model_path}")

✅ Best model saved at: bidirectional_lstm_best_model.h5


In [ ]:
from tensorflow.keras.models import load_model

# Load the model
loaded_model = load_model(model_path)

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


# Helper Function to predict texts

In [ ]:
def predict_sentiment(text, model):
    """
    Predict sentiment for a single text input using a trained Keras LSTM model.
    """
    if isinstance(text, list):
        text = text[0]  # Get the first string if a list is passed

    # Vectorize input
    vectorized_text = text_vectorizer([text])

    # Predict probabilities
    probabilities = model.predict(vectorized_text, verbose=0)[0]

    # Get predicted class index
    predicted_index = int(np.argmax(probabilities))
    print(f"predicted_index (encoded): {predicted_index}") # 2 right now

    # Inverse transform the predicted index to get the original label
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]
    predicted_label_name = reverse_sentiment_map.get(predicted_label, "Unknown")
    return {
        'text': text,
        # 'predicted_class_encoded': predicted_index,
        'predicted_label_original': predicted_label,
        'predicted_label_name': predicted_label_name,
        'probabilities': {
            label_encoder.classes_[i]: float(prob) for i, prob in enumerate(probabilities)
        }
    }

#function to use foundation model
def predict_sentiment_sklearn(text, model, reverse_sentiment_map):
    """
    Predict sentiment for a single text input using a trained scikit-learn model pipeline.
    """
    if isinstance(text, list):
        text = text[0]  # Get the first string if a list is passed

    # The scikit-learn model pipeline likely includes the vectorizer,
    # so we can directly use the pipeline's predict method.
    # It expects a list of strings as input.
    predictions = model.predict([text])
    predicted_index = predictions[0] # The prediction is the class label

    # Map the numerical prediction back to the sentiment label
    predicted_label_name = reverse_sentiment_map.get(predicted_index, "Unknown")

    # If the model supports predict_proba, get probabilities
    probabilities = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba([text])[0]
        # We need the original class labels (-1, 0, 1) from the model's classes_ attribute
        # and map them to the probabilities
        proba_dict = {model.classes_[i]: float(p) for i, p in enumerate(proba)}
        probabilities = proba_dict


    return {
        'text': text,
        'predicted_label': predicted_label_name,
        'predicted_class': predicted_index,
        'probabilities': probabilities
    }
reverse_sentiment_map = {v: k for k, v in sentiment_map.items()}


In [ ]:
# New text sample
new_texts = ["မတန်ဘူးဗျ ၄၀၀၀နဲ့ ထည့်ပေးတဲ့ဟင်းနှစ်မျိုးက နည်းနည်းလေး အရင်တည်းက နည်းတာမဖြူဆိုင်က အရင်ကတော့ ၁၆၀၀နဲ့တန်တယ် အခုတော့မတန်တော့ဘူး"]  # Burmese example


# Model Validation
## Load Foundation model from WandB

In [ ]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('W&B_ACCESS_KEY')) # Login

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayenyeinsan2904 (ayenyeinsan2904-chiang-mai-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Downloading foundation model artifact
run = wandb.init()
artifact = run.use_artifact('ayenyeinsan2904-chiang-mai-university/model-registration-example/foundation_model_myansen:v0', type='model')
artifact_dir = artifact.download()

print(f"Artifact downloaded to: {artifact_dir}")

wandb:   2 of 2 files downloaded.  


Artifact downloaded to: /content/artifacts/foundation_model_myansen:v0


In [ ]:
#loading foundation model
foundation_model = joblib.load(f"{artifact_dir}/best_model_logistic_regression_bow_20250615_133439.pkl")


#Compare the accuracy


In [ ]:
#foundation model accuracy
import json

with open(f"{artifact_dir}/model_metadata_20250615_133439.json", 'r') as f:
    metadata = json.load(f)

foundation_test_accuracy = metadata.get('test_accuracy')

print(f"Foundation model test accuracy: {foundation_test_accuracy}")

Foundation model test accuracy: 0.5416666666666666


In [ ]:
# Current Model Accuracy
current_test_accuracy = best_model_results['Test_Accuracy']
print(f"Current model test accuracy: {current_test_accuracy}")

Current model test accuracy: 0.5


#Model Registration into WandB
## Register the best model into WandB


In [ ]:
if (foundation_test_accuracy > current_test_accuracy):
  print("Foundation model is better")

  # Use the scikit-learn prediction function with the previous model
  result_sklearn = predict_sentiment_sklearn(
      text=new_texts,
      model=foundation_model,
      reverse_sentiment_map=reverse_sentiment_map
  )
  print(result_sklearn)
else:
  print("Current model is better")
  print("LSTM architecture")

  #register into W&B
  best_model_artifact = wandb.Artifact(name='current_best_model', type='model')
  best_model_artifact.add_file(model_path)
  wandb.log_artifact(best_model_artifact)
  wandb.finish()
  print("Registered current best model to W&B🫆")
  # result = predict_sentiment(
  #   text=new_texts,
  #   model=loaded_model
  # )
  # print(result)


Foundation model is better
{'text': 'မတန်ဘူးဗျ ၄၀၀၀နဲ့ ထည့်ပေးတဲ့ဟင်းနှစ်မျိုးက နည်းနည်းလေး အရင်တည်းက နည်းတာမဖြူဆိုင်က အရင်ကတော့ ၁၆၀၀နဲ့တန်တယ် အခုတော့မတန်တော့ဘူး', 'predicted_label': 'negative', 'predicted_class': np.int64(-1), 'probabilities': {np.int64(-1): 0.9471090775462873, np.int64(0): 0.019489326883384513, np.int64(1): 0.0334015955703282}}


In [ ]:

#load the current best model to use back in the pipeline